In [ ]:
!pip install torch transformers scikit-learn numpy ipywidgets -q

In [ ]:
!pip install bitsandbytes accelerate -q

In [ ]:
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.decomposition import PCA
from typing import List, Optional, Tuple
from dataclasses import dataclass
import warnings
warnings.filterwarnings('ignore')

# device verification
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
@dataclass
class ControlAxis:
    """Represents a discovered control dimension"""
    label: str
    vector: np.ndarray
    positive_example: str
    negative_example: str
    layer_idx: int

In [ ]:
from transformers import BitsAndBytesConfig

class SemanticControlSystem:
    def __init__(
        self,
        model_id="Qwen/Qwen2.5-3B-Instruct",
        labeler_id="Qwen/Qwen2.5-7B-Instruct",
        target_layer=18,
        device=device
    ):
        print(f"\n Initializing Hybrid System on {device}...\n")
        self.device = device
        self.target_layer = target_layer

        # Main Model
        print(f" Loading Engine: {model_id}...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_id)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_id,
            torch_dtype=torch.float16,
            device_map="auto"
        )
        self.model.eval()

        # Labeler Model
        print(f"\n Loading Labeler: {labeler_id} (4-bit compressed)...")
        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16
        )

        self.labeler_tokenizer = AutoTokenizer.from_pretrained(labeler_id)
        self.labeler_model = AutoModelForCausalLM.from_pretrained(
            labeler_id,
            quantization_config=quantization_config,
            device_map="auto"
        )
        self.labeler_model.eval()
        print("Hybrid setup complete!\n")

        self.variation_cache = {}

    def generate_variations(
        self,
        prompt: str,
        num_variations: int = 2,
        temperature: float = 1.2,
        max_length: int = 200
    ) -> List[str]:
        """Generate diverse variations to explore latent space"""
        print(f"Generating {num_variations} variations...")

        variations = []
        messages = [
            {"role": "system", "content": "You are a helpful writing assistant."},
            {"role": "user", "content": prompt}
        ]

        input_text = self.tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = self.tokenizer(input_text, return_tensors="pt").to(self.device)

        for i in range(num_variations):
            with torch.no_grad():
                outputs = self.model.generate(
                    **inputs,
                    max_new_tokens=max_length,
                    temperature=temperature,
                    do_sample=True,
                    top_p=0.95,
                    pad_token_id=self.tokenizer.eos_token_id
                )

            generated_text = self.tokenizer.decode(
                outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True
            )
            variations.append(generated_text.strip())
            print(f"Generated variation {i + 1}/{num_variations}")

        return variations

    def extract_activations(
        self,
        texts: List[str],
        layer_idx: Optional[int] = None
    ) -> np.ndarray:
        """Extract hidden state activations from specified layer"""
        if layer_idx is None:
            layer_idx = self.target_layer

        print(f"Extracting activations from layer {layer_idx}...")
        activations = []

        for text in texts:
            messages = [
                {"role": "system", "content": "You are a helpful writing assistant."},
                {"role": "user", "content": text}
            ]
            input_text = self.tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )
            inputs = self.tokenizer(input_text, return_tensors="pt").to(self.device)

            with torch.no_grad():
                outputs = self.model(
                    **inputs,
                    output_hidden_states=True,
                    return_dict=True
                )

            hidden_state = outputs.hidden_states[layer_idx][0, -1, :]
            activations.append(hidden_state.cpu().numpy())

        print("Activations extracted")
        return np.array(activations)

    def perform_pca(
        self,
        activations: np.ndarray,
        n_components: int = 3
    ) -> Tuple[PCA, np.ndarray]:
        """Perform PCA to find principal axes of variation"""
        print(f"Running PCA to extract {n_components} components...")

        pca = PCA(n_components=n_components)
        transformed = pca.fit_transform(activations)

        print(f"Explained variance: {[f'{v:.1%}' for v in pca.explained_variance_ratio_]}")
        return pca, transformed

    def label_axis(
        self,
        positive_text: str,
        negative_text: str
    ) -> str:
        """Label axis using TinyLlama or heuristics"""
        if self.labeler_model is not None:
            return self._label_with_local_model(positive_text, negative_text)
        return self._fallback_labeling(positive_text, negative_text)

    def _label_with_local_model(self, positive_text: str, negative_text: str) -> str:
        prompt = f"""Compare these two sentences:
Sentence 1: "{positive_text}"
Sentence 2: "{negative_text}"

Choose the ONE word from this list that best describes the main difference in style, tone, or content:
[Formality, Enthusiasm, Politeness, Detail, Urgency, Professionalism, Negativity, Confidence, Verbosity]

Output ONLY the one word."""

        messages = [
            {"role": "system", "content": "You are a linguistic expert classifying text styles."},
            {"role": "user", "content": prompt}
        ]

        input_text = self.labeler_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = self.labeler_tokenizer(input_text, return_tensors="pt").to(self.device)

        try:
            with torch.no_grad():
                outputs = self.labeler_model.generate(
                    **inputs,
                    max_new_tokens=5,
                    temperature=0.1,
                    do_sample=False
                )

            generated = self.labeler_tokenizer.decode(
                outputs[0][inputs['input_ids'].shape[1]:],
                skip_special_tokens=True
            ).strip()

            import re
            words = re.findall(r'[A-Za-z]+', generated)
            if words:
                return words[0].capitalize()

            return "Style"

        except Exception as e:
            print(f"Labeling error: {e}")
            return "Style"

    def _fallback_labeling(self, positive_text: str, negative_text: str) -> str:
        """Heuristic-based labeling"""
        pos_len = len(positive_text.split())
        neg_len = len(negative_text.split())

        if abs(pos_len - neg_len) > 15:
            return "Verbosity"
        elif abs(positive_text.count('!') - negative_text.count('!')) > 2:
            return "Enthusiasm"
        elif abs(positive_text.count('?') - negative_text.count('?')) > 2:
            return "Inquisitiveness"

        formal_words = ['however', 'therefore', 'furthermore', 'moreover']
        informal_words = ['yeah', 'cool', 'hey', 'ok']

        pos_formal = sum(1 for w in formal_words if w in positive_text.lower())
        neg_formal = sum(1 for w in formal_words if w in negative_text.lower())

        if abs(pos_formal - neg_formal) > 1:
            return "Formality"

        return "Style"

    def discover_axes(
        self,
        prompt: str,
        num_variations: int = 5,
        n_axes: int = 4
    ) -> List[ControlAxis]:
        """Full pipeline: Generate, extract, PCA, label"""
        print("\n" + "="*60)
        print("DISCOVERING CONTROL AXES")
        print("="*60 + "\n")

        # Generate variations
        variations = self.generate_variations(prompt, num_variations)
        self.last_variations = variations  # 👈 SAVED FOR GRAPH

        # Extract activations
        activations = self.extract_activations(variations)

        # PCA
        pca, transformed = self.perform_pca(activations, n_components=n_axes)
        self.last_transformed = transformed  # 👈 SAVED FOR GRAPH

        # Labels
        axes = []
        print("\n Labeling axes...")
        for i in range(n_axes):
            component_scores = transformed[:, i]
            pos_idx = np.argmax(component_scores)
            neg_idx = np.argmin(component_scores)

            pos_text = variations[pos_idx]
            neg_text = variations[neg_idx]

            label = self.label_axis(pos_text, neg_text)

            vector = pca.components_[i]

            # If the label is about length, ensure Positive = Longer
            if label.lower() in ["verbosity", "length", "detail"]:
                if len(pos_text) < len(neg_text):
                    vector = vector * -1
                    pos_text, neg_text = neg_text, pos_text

            print(f"   Axis {i+1}: {label}")

            axis = ControlAxis(
                label=label,
                vector=vector,
                positive_example=pos_text,
                negative_example=neg_text,
                layer_idx=self.target_layer
            )
            axes.append(axis)

        self.last_axes = axes

        print("\n" + "="*60)
        print(f"Discovered {len(axes)} control axes!")
        print("="*60 + "\n")

        return axes

    def steer_generation(
        self,
        prompt: str,
        axis: ControlAxis,
        coefficient: float = 1.0,
        max_new_tokens: int = 150
    ) -> str:
        """Generate text with dynamic multi-layer activation steering"""
        messages = [
            {"role": "system", "content": "You are a helpful writing assistant."},
            {"role": "user", "content": prompt}
        ]
        input_text = self.tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = self.tokenizer(input_text, return_tensors="pt").to(self.device)

        # Automatically find the total layers of whatever model is loaded
        total_layers = self.model.config.num_hidden_layers

        # the middle 20% of the network (e.g., 40% to 60% depth)
        layer_start = int(total_layers * 0.4)
        layer_end = int(total_layers * 0.6)
        num_layers = layer_end - layer_start + 1

        steering_vector = torch.tensor(
            (axis.vector * coefficient) / num_layers,
            dtype=torch.float16 if self.device == "cuda" else torch.float32,
            device=self.device
        )

        def steering_hook(module, input, output):
            if isinstance(output, tuple):
                hidden_states = output[0]
            else:
                hidden_states = output

            # Add the fractional vector to the final token's hidden state
            hidden_states[:, -1, :] += steering_vector

            if isinstance(output, tuple):
                return (hidden_states,) + output[1:]
            return hidden_states

        # Register the hook on calculated layers
        handles = []
        for l in range(layer_start, layer_end + 1):
            layer = self.model.model.layers[l]
            handles.append(layer.register_forward_hook(steering_hook))

        try:
            with torch.no_grad():
                outputs = self.model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    do_sample=True,
                    temperature=0.2,
                    top_p=0.9,
                    pad_token_id=self.tokenizer.eos_token_id
                )

            generated_text = self.tokenizer.decode(
                outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True
            )
        finally:
            # Remove hooks to stop memory leaks
            for handle in handles:
                handle.remove()

        return generated_text.strip()

print("SemanticControlSystem class defined!")

In [ ]:
!pip install -U bitsandbytes>=0.46.1 -q
system = SemanticControlSystem(
    model_id="Qwen/Qwen2.5-3B-Instruct",
    labeler_id="Qwen/Qwen2.5-7B-Instruct"
)

In [ ]:
prompt = "Write a one-sentence text message to my friend canceling on their party tonight."

axes = system.discover_axes(
    prompt=prompt,
    num_variations=5,
    n_axes=4
)

print("\n DISCOVERED AXES:\n")
for i, axis in enumerate(axes, 1):
    display_label = axis.label if axis.label != "Text" else f"Style {i}"

    print(f"{i}. {display_label}")
    print(f"Positive: {axis.positive_example}")
    print(f"Negative: {axis.negative_example}")
    print()

In [ ]:
import time

print("Exp 1: Coefficient Breaking Point\n")

prompt = "Explain the rules of boxing in one sentence."
test_coefficients = [20.0, 40.0, 60.0, 80.0, 100.0, 150.0]

# first axis discovered earlier
test_axis = system.last_axes[0]
print(f"Target Concept: {test_axis.label}\n")
print("-" * 50)

for coef in test_coefficients:
    print(f"Slider Value: +{coef}")

    steered_text = system.steer_generation(
        prompt=prompt,
        axis=test_axis,
        coefficient=coef,
        max_new_tokens=40
    )

    print(f"Generated Text: {steered_text}")
    print("-" * 50)

In [ ]:
import time

print(" Exp 1 (again): Bidirectional Generalization\n")

# Discover a Universal Axis
discovery_prompt = "Explain an API."
print(f" Discovering axes on: '{discovery_prompt}'")
system.discover_axes(prompt=discovery_prompt, num_variations=5, n_axes=4)

# Take strongest axis
test_axis = system.last_axes[0]
print(f"\n Target Concept: {test_axis.label}")
print("=" * 70)

# Grid Search w/ diff domains
test_prompts = [
    "Explain the rules of boxing.",              # Sports
    "Write a python function to reverse a string.",  # Code
    "Give me advice on negotiating a salary."        # Professional
]

# negative + positive coeffs
test_coefficients = [-150.0, -100.0, -50.0, -25.0, 0.0, 25.0, 50.0, 100.0, 150.0]

for prompt in test_prompts:
    print(f"\ Testing prompt: {prompt}")
    print("-" * 70)

    for coef in test_coefficients:
        steered_text = system.steer_generation(
            prompt=prompt,
            axis=test_axis,
            coefficient=coef,
            max_new_tokens=100
        )

        print(f"[{coef:>6}] : {steered_text.strip().replace(chr(10), ' ')}")
    print("=" * 70)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

print("Generating UIST-Ready Trade-off Graph...")
coefficients = [-150, -100, -50, -25, 0, 25, 50, 100, 150]
fluency_scores = [0, 2, 10, 10, 10, 10, 10, 4, 0]

steering_success = [0, 1, 3, 5, 5, 7, 9, 3, 0]

plt.style.use('default')
fig, ax = plt.subplots(figsize=(10, 6), dpi=300) # High DPI for paper printing

ax.plot(coefficients, fluency_scores, marker='o', markersize=8, linewidth=2.5,
        label='Grammatical Fluency', color='#2ca02c') # Green

ax.plot(coefficients, steering_success, marker='s', markersize=8, linewidth=2.5,
        label='Steering Success (Detail Level)', color='#1f77b4', linestyle='--') # Blue Dashed

ax.axvspan(-50, 50, color='gray', alpha=0.15, label='Optimal UI Bounds (Safe Zone)')

# Pointing out the Chinese charactrs hallucination
ax.annotate('Bilingual Bleed\n(Hallucination)', xy=(-100, 2), xytext=(-140, 4.5),
            arrowprops=dict(facecolor='black', shrink=0.05, width=1.5, headwidth=7),
            fontsize=10, fontweight='bold', color='darkred')

ax.annotate('Semantic Loop\n(Degradation)', xy=(100, 4), xytext=(105, 6.5),
            arrowprops=dict(facecolor='black', shrink=0.05, width=1.5, headwidth=7),
            fontsize=10, fontweight='bold', color='darkred')

ax.set_title('Trade-off Between Fluency and Steering Intensity',
             fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Steering Coefficient', fontsize=12, fontweight='bold')
ax.set_ylabel('Evaluation Score (0 - 10)', fontsize=12, fontweight='bold')

ax.set_xticks(coefficients)
ax.set_yticks(range(0, 12, 2))
ax.grid(True, linestyle=':', alpha=0.7)

ax.legend(loc='lower center', fontsize=11, framealpha=0.9)

plt.tight_layout()
plt.savefig('uist_evaluation_graph.png', format='png', bbox_inches='tight')
print("Saved as 'uist_evaluation_graph.png' in your Colab files!")
plt.show()

In [ ]:
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
import numpy as np
import torch
import re

def visualize_latent_space(system):
    print("Building 3D Graph from the latest Discovery math...")

    #grab the exact math, texts, and labels from the last run
    variations = system.last_variations
    transformed = system.last_transformed
    axes = system.last_axes

    axis_1_label = axes[0].label

    # Categorize every single point to build the Legend
    print("Categorizing points for the legend...")
    categories = []

    for text in variations:
        cat_prompt = f"Categorize the tone or genre of this text in exactly ONE word:\n\"{text}\""
        messages = [
            {"role": "system", "content": "You are a classifier. Output only one word."},
            {"role": "user", "content": cat_prompt}
        ]
        input_text = system.labeler_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = system.labeler_tokenizer(input_text, return_tensors="pt").to(system.device)

        with torch.no_grad():
            out = system.labeler_model.generate(**inputs, max_new_tokens=4, do_sample=False)

        cat_raw = system.labeler_tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

        words = re.findall(r'[A-Za-z]+', cat_raw)
        cat_clean = words[0].capitalize() if words else "Unknown"
        categories.append(cat_clean)

    df = pd.DataFrame({
        'PC1 (Axis 1)': transformed[:, 0],
        'PC2 (Axis 2)': transformed[:, 1],
        'PC3 (Axis 3)': transformed[:, 2],
        'Text': [v[:60] + "..." for v in variations],
        'Category': categories,
        'Full_Text': variations
    })

    fig = px.scatter_3d(
        df, x='PC1 (Axis 1)', y='PC2 (Axis 2)', z='PC3 (Axis 3)',
        hover_name='Text',
        color='Category',
        title=f"3D Latent Space (Primary Axis: {axis_1_label})"
    )

    fig.update_traces(marker=dict(size=8, line=dict(width=2, color='DarkSlateGrey')))

    fig.add_trace(go.Scatter3d(
        x=[df['PC1 (Axis 1)'].min(), df['PC1 (Axis 1)'].max()],
        y=[0, 0],
        z=[0, 0],
        mode='lines',
        line=dict(color='red', width=5),
        name=f'Steering Vector ({axis_1_label})'
    ))

    fig.update_layout(
        scene=dict(
            xaxis_title=f'Axis 1 ({axis_1_label})',
            yaxis_title='Axis 2',
            zaxis_title='Axis 3'
        ),
        width=900,
        height=700,
        margin=dict(l=0, r=0, b=0, t=40)
    )

    fig.show()

visualize_latent_space(system)

In [ ]:
# Install server and tunneling libraries
!pip install fastapi uvicorn nest-asyncio pydantic -q
!npm install -g localtunnel -q

from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import uvicorn
import nest_asyncio
import threading
import subprocess
import time

# Define the data structures
class DiscoverRequest(BaseModel):
    prompt: str
    num_variations: int = 5
    n_axes: int = 4

class SteerRequest(BaseModel):
    prompt: str
    axis_index: int
    coefficient: float
    max_tokens: int = 150

# Initialize the API
app = FastAPI(title="Latent-Steer API")

# Allow the React app to talk to this Colab server
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

global_axes = []

# Define the Endpoints
@app.post("/discover")
async def discover_endpoint(req: DiscoverRequest):
    global global_axes
    print(f"\n API Request: Discovering axes for -> '{req.prompt}'")

    global_axes = system.discover_axes(
        prompt=req.prompt,
        num_variations=req.num_variations,
        n_axes=req.n_axes
    )

    response_data = []
    for i, axis in enumerate(global_axes):
        response_data.append({
            "index": i,
            "label": axis.label,
            "positive_example": axis.positive_example,
            "negative_example": axis.negative_example
        })

    return {"status": "success", "axes": response_data}

@app.post("/steer")
async def steer_endpoint(req: SteerRequest):
    if not global_axes or req.axis_index >= len(global_axes):
        return {"error": "Invalid axis. Please run /discover first."}

    target_axis = global_axes[req.axis_index]
    print(f"\n API Request: Steering {target_axis.label} to {req.coefficient}")

    generated_text = system.steer_generation(
        prompt=req.prompt,
        axis=target_axis,
        coefficient=req.coefficient,
        max_new_tokens=req.max_tokens
    )

    return {"status": "success", "generated_text": generated_text}

# Launch the Server
nest_asyncio.apply()

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")

server_thread = threading.Thread(target=run_server)
server_thread.start()

# Create a public tunnel to bypass VS Code port forwarding restrictions
print("Creating public tunnel...")
time.sleep(3)
process = subprocess.Popen(["lt", "--port", "8000"], stdout=subprocess.PIPE)
for line in process.stdout:
    print(f"\n YOUR PUBLIC API URL: {line.decode().strip()}\n")
    break

In [ ]:
!curl ipv4.icanhazip.com

In [ ]:
import requests
url = "http://127.0.0.1:8000"
print(requests.get(url + "/docs").status_code)
